# infinaxis controls demo

This notebook shows the main `Axis` and `GcodeControls` options that affect generated gcode.

The next cell contains reusable functions that help keep subsequent cells concise

In [ ]:
import lab.fullcontrol.infinaxis as fci

EW = 0.6
EH = 0.3
print_settings = {'extrusion_width': EW, 'extrusion_height': EH}

base_steps_data = [
    dict(x=0, y=10, z=EH / 2, b=0, c=0),
    dict(x=4, y=10, z=EH / 2, b=10, c=20),
    dict(x=8, y=12, z=EH, b=20, c=40),
    dict(x=12, y=14, z=EH * 1.5, b=30, c=60),
]

def build_steps(head_chain, bed_chain, steps_data=base_steps_data):
    Point = fci.configure_point(head_chain, bed_chain)
    return [Point(**step_data) for step_data in steps_data]

def controls(head_chain, bed_chain, **kwargs):
    return fci.GcodeControls(
        head_chain=head_chain,
        bed_chain=bed_chain,
        initialization_data=print_settings,
        **kwargs,
    )

def print_gcode_sample(label, steps, gcode_controls, look_for):
    gcode = fci.transform(steps, 'gcode', gcode_controls)
    print('___\n' + label)
    print(f'Look for: {look_for}')
    print('\n'.join(gcode.split('\n')[-8:]))


## Baseline

A normal two-rotary-axis example. `Axis.name` is the gcode name and, for standard `A/B/C`, also provides the default kinematic `type`.

In [ ]:
head_chain = [fci.Axis(name='B')]
bed_chain = [fci.Axis(name='C')]
steps = build_steps(head_chain, bed_chain)

print_gcode_sample(
    'baseline Axis(name="B") + Axis(name="C"):',
    steps,
    controls(head_chain, bed_chain),
    'B and C appear in each move; XYZ is inverse-kinematics output.',
)


## Axis.name and Axis.type

Use `name` for the emitted gcode axis and `type` for the kinematic behavior. This allows machine names like `CI` and `CII` to both behave as C-style rotary axes.

In [ ]:
head_chain = [fci.Axis(name='CII', type='C')]
bed_chain = [fci.Axis(name='CI', type='C')]
Point = fci.configure_point(head_chain, bed_chain)
steps = [
    Point(x=0, y=10, z=EH / 2, CI=0, CII=0),
    Point(x=4, y=10, z=EH / 2, CI=20, CII=-10),
    Point(x=8, y=12, z=EH, CI=40, CII=-20),
]

print_gcode_sample(
    'custom axis names with shared C kinematics:',
    steps,
    controls(head_chain, bed_chain),
    'CI and CII are emitted, while type="C" controls the rotation math.',
)


## Axis.orientation

`orientation` flips the sign used by the kinematic calculation. The commanded axis value is still emitted with the configured `Axis.name`; the XYZ solution changes.

In [ ]:
orientation_steps_data = [
    dict(x=0, y=10, z=EH / 2, c=0),
    dict(x=4, y=10, z=EH / 2, c=20),
    dict(x=8, y=12, z=EH, c=40),
]

for orientation in [1, -1]:
    head_chain = []
    bed_chain = [fci.Axis(name='C', orientation=orientation)]
    steps = build_steps(head_chain, bed_chain, orientation_steps_data)
    print_gcode_sample(
        f'Axis(name="C", orientation={orientation}):',
        steps,
        controls(head_chain, bed_chain),
        'Compare XYZ values between orientation=1 and orientation=-1. C values remain commanded values.',
    )


## Axis.offset

`offset` describes the location of an axis relative to the previous axis in the chain. An accurate measurement of the offset is paramount for the inverse kinematics to function as intented. On common example of this offset is a tilting nozzle where the offset will be the distance between the tilting axis and the tip of the nozzle.

In [ ]:
for offset in [fci.Point(x=0, y=0, z=0), fci.Point(x=5, y=0, z=0)]:
    head_chain = [fci.Axis(name='B', offset=offset)]
    bed_chain = [fci.Axis(name='C')]
    steps = build_steps(head_chain, bed_chain)
    print_gcode_sample(
        f'Axis(name="B", offset={offset}):',
        steps,
        controls(head_chain, bed_chain),
        'Compare XYZ values as the head rotary axis offset changes.',
    )


## GcodeControls.verbose

`verbose=True` adds distance diagnostics as gcode comments.

In [ ]:
head_chain = [fci.Axis(name='B')]
bed_chain = [fci.Axis(name='C')]
steps = build_steps(head_chain, bed_chain)

for verbose in [False, True]:
    print_gcode_sample(
        f'GcodeControls(verbose={verbose}):',
        steps,
        controls(head_chain, bed_chain, verbose=verbose),
        'Verbose output adds ; distance and system comments when enabled.',
    )


## GcodeControls.planning_axes_mono

`planning_axes_mono=True` emits a synthetic axis value based on accumulated model/path distance. The name of this axis value is defined by setting `GcodeControls.planning_axes_names`. This can be used on compatible controller boards to effectively overwrite the firmware distance calculation done during motion planning by configuring the printer to use the synthetic planning axis value as its only axes contributing to distance. In RepRap firmware this can be done by setting the planning axis to be the only "linear" axis, and all the real axes to "rotational". This has been tested on Duet 2 and Duet 3 boards.

In [ ]:
head_chain = [fci.Axis(name='B')]
bed_chain = [fci.Axis(name='C')]
steps = build_steps(head_chain, bed_chain)

print_gcode_sample(
    'GcodeControls(planning_axes_mono=True):',
    steps,
    controls(head_chain, bed_chain, planning_axes_mono=True, planning_axes_names=["U"]),
    'U appears after B/C and increases as accumulated distance changes.',
)


 ## GcodeControls.planning_axes_tripple

`planning_axes_tripple=True` emits the model-space `X/Y/Z` values before applying the inverse kinematics transformation. These can be used to allow conventional 3 axis motion planners to effectively see the model space and hence plan based on it as opposed to the transformed system space, where the 3-axis distance is not readily available. The axes name for the emitted model-space coordinates is also set using `GcodeControls.planning_axes_names`. This can be used in a similair manner to the mono planning axis, by setting the 3 planning axes to "linear" and the real axes to "rotational". This has been tested on RepRap firmware running on a Duet 3 board.

This tripple planning axes approach should be compatible with Klipper firmware since the axes considered for the distance calculation is hardcoded at the 3 first ones (at the time of writing this). The mono planning axis approach should hence not be compatible. Please note that this has not been tested on a printer running Klipper, only on RepRap firmware!

In [ ]:
head_chain = [fci.Axis(name='B')]
bed_chain = [fci.Axis(name='C')]
steps = build_steps(head_chain, bed_chain)

print_gcode_sample(
    'GcodeControls(planning_axes_tripple=True):',
    steps,
    controls(head_chain, bed_chain, planning_axes_tripple=True, planning_axes_names=["U","V","W"]),
    'U/V/W show the model-space x/y/z values alongside transformed XYZ/B/C.',
)


## GcodeControls.inverse_time_feedrate

`inverse_time_feedrate=True` changes the F calculation for extrusion moves to run in inverse time feedrate mode. This is commonly activated by using `G93` in the gcode. Infinaxis is currently set up to call `G93` at the start of the gcode and then `G94` at the end to set the printer back to unit pr minute mode (default).

In [ ]:
head_chain = [fci.Axis(name='B')]
bed_chain = [fci.Axis(name='C')]
steps = build_steps(head_chain, bed_chain)

print_gcode_sample(
    'GcodeControls(inverse_time_feedrate=True):',
    steps,
    controls(head_chain, bed_chain, inverse_time_feedrate=True),
    'F values differ from the baseline calculation.',
)


## GcodeControls.xyz_orientation

`xyz_orientation` flips output signs for system XYZ axes. This is important of extra axes are added to a printer which doesnt inherently follow the right-hand rule, and provides an alternative to changing the configuration of the printer.

In [ ]:
head_chain = [fci.Axis(name='B')]
bed_chain = [fci.Axis(name='C')]
steps = build_steps(head_chain, bed_chain)

print_gcode_sample(
    'Baseline output for regular orientation, GcodeControls(xyz_orientation=[1, 1, 1]):',
    steps,
    controls(head_chain, bed_chain, xyz_orientation=[1, 1, 1]),
    '',
)

print_gcode_sample(
    'X orientation flipped, GcodeControls(xyz_orientation=[-1, 1, 1]):',
    steps,
    controls(head_chain, bed_chain, xyz_orientation=[-1, 1, 1]),
    'X output changes sign compared with the baseline.',
)
